<a href="https://colab.research.google.com/github/helenlu-vbs/Strategic-International-Finance-Bootcamp-Vlerick-/blob/main/Comparables_GBM_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
!pip install -q lightgbm shap openpyxl
import pandas as pd

In [56]:
# ============================================================
# 2. Imports
# ============================================================
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings("ignore")

In [57]:
# ============================================================
# 3. User settings
# ============================================================
MAIN_CSV = "/content/compustat_combined_USD_with_refinitiv.csv"
COMPS_XLSX = "/content/Galata_wind_comps.xlsx"

OUTPUT_DIR = Path("/content/galata_lgbm_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
VALID_SIZE = 0.20
MIN_TRAIN_OBS = 200

# Hard-coded Galata name
TARGET_COMPANY_NAME = "Galata Wind Enerji Anonim Sirket"

# ------------------------------------------------------------
# Exact columns based on your headings
# ------------------------------------------------------------
COMPANY_COL = "Company Common Name"
COUNTRY_COL = "Country of Headquarters"
INDUSTRY_COL = "TRBC Industry Name"

# Replace these three if your actual target column names differ
EVSALES_TARGET = "EV2Sales"
EVEBITDA_TARGET = "EV2EBITDA"
PE_TARGET = "PE"

TARGET_COLS = [EVSALES_TARGET, EVEBITDA_TARGET, PE_TARGET]

# Exact numeric feature columns from your message
NUMERIC_FEATURES = [
    "Sales",
    "SaleGrwth",
    "ROA",
    "ATR",
    "EBITDA_PCT",
    "Capex2EBITDA",
    "NetDebt2TotalAssets",
    "OpLev",
    "Cash2TotalAssets",
    "Rnd2Sale",
    "TaxRate",
    "PayoutRatio",
    'Total CO2 Equivalent Emissions To EVIC USD in million',
    'ESG Controversies Score',
    'ESG Score',
    "CDS",
]

CATEGORICAL_FEATURES = [
    INDUSTRY_COL,
]

In [58]:
# ============================================================
# 4. Helper functions
# ============================================================
def slugify(x: str) -> str:
    x = re.sub(r"[^A-Za-z0-9]+", "_", str(x)).strip("_")
    return x[:120] if x else "unknown"


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def load_peer_names_from_excel(xlsx_path: str, company_col: str):
    """
    Read the Galata comp sheet and extract the company names.
    """
    peers_df = pd.read_excel(xlsx_path, sheet_name=0)

    if company_col not in peers_df.columns:
        raise ValueError(
            f"'{company_col}' not found in {xlsx_path}. "
            f"Columns found: {list(peers_df.columns)}"
        )

    peer_names = (
        peers_df[company_col]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Drop common summary rows if present
    peer_names = peer_names[
        ~peer_names.str.contains("PEER_MEAN|PEER_MEDIAN|SUMMARY", case=False, na=False)
    ]

    return peer_names.drop_duplicates().tolist()


def check_required_columns(df: pd.DataFrame):
    """
    Make sure all required columns are present in the dataset.
    """
    required = [COMPANY_COL, COUNTRY_COL, INDUSTRY_COL] + NUMERIC_FEATURES + TARGET_COLS
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(
            "These required columns are missing from the CSV:\n"
            f"{missing}\n\n"
            f"Available columns are:\n{list(df.columns)}"
        )


def prepare_dataset(df: pd.DataFrame):
    """
    Convert numeric columns to numeric and categorical columns to category dtype.
    """
    out = df.copy()

    for col in NUMERIC_FEATURES + TARGET_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].astype("string").fillna("__MISSING__").astype("category")

    out[COMPANY_COL] = out[COMPANY_COL].astype(str).str.strip()

    return out


def make_test_mask(df: pd.DataFrame, peer_names: list):
    """
    Mark all firms from the Galata comp sheet as test observations.
    """
    return df[COMPANY_COL].astype(str).str.strip().isin(peer_names)


def build_feature_list_for_target(target_col: str):
    """
    Build the feature list for one target.

    Rules:
    - use Country of Headquarters and TRBC Industry Name as categorical
    - use the numeric features listed above
    - do NOT use Company Common Name
    - do NOT use other valuation multiples as features
    """
    features = CATEGORICAL_FEATURES + NUMERIC_FEATURES
    return features

In [59]:
# ============================================================
# 5. Train one LightGBM model for one target
# ============================================================
def train_one_target(
    df_full: pd.DataFrame,
    test_mask: pd.Series,
    target_col: str,
):
    """
    Train LightGBM on all non-test observations.

    - validation split: 80/20 on remaining data
    - early stopping after 10 rounds
    - target transformed to ln(target)
    - at least 200 usable training rows required
    """

    print("\n" + "=" * 90)
    print(f"Training model for: {target_col}")
    print("=" * 90)

    feature_cols = build_feature_list_for_target(target_col)

    # --------------------------------------------------------
    # Use only non-test rows for training pool
    # Require strictly positive target because of ln(target)
    # --------------------------------------------------------
    train_pool = df_full.loc[~test_mask].copy()
    train_pool = train_pool[train_pool[target_col] > 0].copy()

    # Keep only rows with all categorical vars present
    # Numeric missing values can be handled by LightGBM
    for c in CATEGORICAL_FEATURES:
        train_pool = train_pool[train_pool[c].notna()]

    if len(train_pool) < MIN_TRAIN_OBS:
        raise ValueError(
            f"Target '{target_col}' has only {len(train_pool)} usable training observations. "
            f"Need at least {MIN_TRAIN_OBS}."
        )

    # --------------------------------------------------------
    # Drop feature columns with no variation in training data
    # --------------------------------------------------------
    usable_features = []
    for col in feature_cols:
        if train_pool[col].nunique(dropna=False) > 1:
            usable_features.append(col)

    X = train_pool[usable_features].copy()
    y = np.log(train_pool[target_col].astype(float))

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y,
        test_size=VALID_SIZE,
        random_state=RANDOM_STATE
    )

    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=[c for c in CATEGORICAL_FEATURES if c in usable_features],
        free_raw_data=False
    )

    valid_data = lgb.Dataset(
        X_valid,
        label=y_valid,
        categorical_feature=[c for c in CATEGORICAL_FEATURES if c in usable_features],
        free_raw_data=False
    )

    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "learning_rate": 0.05,
        "num_leaves": 31,
        "min_data_in_leaf": 200,
        "verbose": -1,
        "seed": RANDOM_STATE,
    }

    model = lgb.train(
        params=params,
        train_set=train_data,
        valid_sets=[train_data, valid_data],
        valid_names=["train", "valid"],
        num_boost_round=100,
        callbacks=[
            lgb.early_stopping(stopping_rounds=10, verbose=True),
            lgb.log_evaluation(period=50),
        ],
    )

    # --------------------------------------------------------
    # Validation metrics on original scale
    # --------------------------------------------------------
    valid_pred_log = model.predict(X_valid, num_iteration=model.best_iteration)
    valid_pred = np.exp(valid_pred_log)
    valid_actual = np.exp(y_valid)

    metrics = {
        "target": target_col,
        "best_iteration": model.best_iteration,
        "n_train": len(X_train),
        "n_valid": len(X_valid),
        "n_features": len(usable_features),
        "valid_rmse_original_scale": rmse(valid_actual, valid_pred),
        "valid_mae_original_scale": mean_absolute_error(valid_actual, valid_pred),
    }

    print("Validation metrics:")
    print(metrics)

    # --------------------------------------------------------
    # Refit the best tuned model on combined train and validation data
    # --------------------------------------------------------
    X_full_train = pd.concat([X_train, X_valid], ignore_index=True)
    y_full_train = pd.concat([y_train, y_valid], ignore_index=True)

    full_train_data = lgb.Dataset(
        X_full_train,
        label=y_full_train,
        categorical_feature=[c for c in CATEGORICAL_FEATURES if c in usable_features],
        free_raw_data=False
    )

    final_model = lgb.train(
        params=params,
        train_set=full_train_data,
        num_boost_round=model.best_iteration, # Use the best iteration found previously
    )

    # --------------------------------------------------------
    # Predict all Galata comp firms = test observations
    # --------------------------------------------------------
    test_df = df_full.loc[test_mask].copy()

    # SHAP only for rows with non-missing features in categorical vars
    X_test = test_df[usable_features].copy()

    pred_log = final_model.predict(X_test, num_iteration=final_model.best_iteration)
    pred = np.exp(pred_log)

    results = pd.DataFrame({
        COMPANY_COL: test_df[COMPANY_COL].values,
        "target_multiple": target_col,
        "predicted_value": pred,
        "predicted_ln_value": pred_log,
    })

    actual = pd.to_numeric(test_df[target_col], errors="coerce")
    results["actual_value"] = actual.values
    results["actual_ln_value"] = np.where(actual > 0, np.log(actual), np.nan)
    results["prediction_error"] = results["predicted_value"] - results["actual_value"]
    results["abs_prediction_error"] = np.abs(results["prediction_error"])


    # --------------------------------------------------------
    # SHAP plots
    # One waterfall plot per test firm
    # --------------------------------------------------------
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer(X_test)

    shap_dir = OUTPUT_DIR / f"shap_{slugify(target_col)}"
    shap_dir.mkdir(parents=True, exist_ok=True)

    for i in range(len(X_test)):
        company_name = str(test_df.iloc[i][COMPANY_COL])
        # Extract the first part of the company name
        first_part_company_name = company_name.split(' ')[0]
        # Revised filename to include first part of company name and target multiple
        file_name = shap_dir / f"{slugify(first_part_company_name)}_{slugify(target_col)}.png"

        plt.figure(figsize=(10, 6))
        # Changed from waterfall to bar for horizontal representation of individual explanations
        shap.plots.bar(shap_values[i], max_display=15, show=False)
        plt.title(f"{target_col} SHAP for: {company_name}")
        plt.tight_layout()
        plt.savefig(file_name, dpi=200, bbox_inches="tight")
        plt.close()

    # Summary SHAP plot across all test firms
    plt.figure(figsize=(10, 6))
    # Added plot_type='bar' for horizontal summary plot
    shap.summary_plot(shap_values.values, X_test, plot_type='bar', show=False, max_display=15)
    plt.title(f"SHAP summary: {target_col}")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"shap_summary_{slugify(target_col)}.png", dpi=200, bbox_inches="tight")
    plt.close()

    # --------------------------------------------------------
    # Save outputs
    # --------------------------------------------------------
    results.to_csv(OUTPUT_DIR / f"predictions_{slugify(target_col)}.csv", index=False)
    pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / f"validation_metrics_{slugify(target_col)}.csv", index=False)

    return results, metrics, fi


In [60]:
# ============================================================
# 6. Load data
# ============================================================
url = "https://raw.githubusercontent.com/helenlu-vbs/Strategic-International-Finance-Bootcamp-Vlerick-/main/compustat_combined_USD_with_refinitiv.csv"
df = pd.read_csv(url)
# Reduce fragmentation post read_stata()
df = df.copy()
check_required_columns(df)
df = prepare_dataset(df)

peer_names = pd.read_csv("https://raw.githubusercontent.com/helenlu-vbs/Strategic-International-Finance-Bootcamp-Vlerick-/refs/heads/main/GalataWindComps.csv")
print(f"Number of names in comp sheet: {len(peer_names)}")



Number of names in comp sheet: 13


In [61]:
all_results = []
all_metrics = []

# Extract peer names from the DataFrame and combine with the target company
peer_names_list = peer_names.iloc[:, 0].dropna().astype(str).str.strip().tolist()
all_test_company_names = list(set([TARGET_COMPANY_NAME] +peer_names_list ))

# Create the test mask for all companies to be predicted
test_mask = make_test_mask(df, all_test_company_names)

for target_col in TARGET_COLS:
    results, metrics, fi = train_one_target(
        df_full=df,
        test_mask=test_mask,
        target_col=target_col,
    )
    all_results.append(results)
    all_metrics.append(metrics)

combined_results = pd.concat(all_results, ignore_index=True)
combined_metrics = pd.DataFrame(all_metrics)

combined_results.to_csv(OUTPUT_DIR / "all_test_predictions_combined.csv", index=False)
combined_metrics.to_csv(OUTPUT_DIR / "all_validation_metrics_combined.csv", index=False)

print("\nDone.")
print(f"Outputs saved to: {OUTPUT_DIR}")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)


Training model for: EV2Sales
Training until validation scores don't improve for 10 rounds
[50]	train's rmse: 0.840531	valid's rmse: 0.891601
[100]	train's rmse: 0.760106	valid's rmse: 0.855603
Did not meet early stopping. Best iteration is:
[100]	train's rmse: 0.760106	valid's rmse: 0.855603
Validation metrics:
{'target': 'EV2Sales', 'best_iteration': 100, 'n_train': 12964, 'n_valid': 3241, 'n_features': 17, 'valid_rmse_original_scale': np.float64(1355.100407521196), 'valid_mae_original_scale': 57.54756399426069}

Training model for: EV2EBITDA
Training until validation scores don't improve for 10 rounds
[50]	train's rmse: 0.796545	valid's rmse: 0.867575
[100]	train's rmse: 0.718415	valid's rmse: 0.830024
Did not meet early stopping. Best iteration is:
[100]	train's rmse: 0.718415	valid's rmse: 0.830024
Validation metrics:
{'target': 'EV2EBITDA', 'best_iteration': 100, 'n_train': 10904, 'n_valid': 2727, 'n_features': 17, 'valid_rmse_original_scale': np.float64(782.0197286610262), 'vali